In [1]:
import os
import glob
import cv2
import numpy as np
import hashlib
from shapely.geometry import Polygon
from collections import Counter, defaultdict
from tqdm import tqdm

# Ganti path dataset sesuai lokasi
root_dir = "C:/AI/sampah_sungai/Dataset_final/Dataset_yolo_Segmentation_v3"

def md5(fname, block_size=65536):
    """
    Menghitung hash MD5 dari file. Berguna untuk mendeteksi duplikat.
    """
    h = hashlib.md5()
    with open(fname, 'rb') as f:
        for chunk in iter(lambda: f.read(block_size), b''):
            h.update(chunk)
    return h.hexdigest()

def parse_label(path):
    """
    Memparsing file label dengan fleksibilitas format.
    
    Format yang diharapkan:
      - Baris dengan format standar: 
          class_id x_center y_center width height [x1 y1 x2 y2 ... xn yn]
        * Nilai diharapkan sudah ternormalisasi antara 0 dan 1.
      - Jika split dengan spasi tidak memberikan token yang cukup (< 5), 
        skrip akan mencoba split dengan koma.
    """
    objs = []
    with open(path, 'r') as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

            # Pertama, coba split menggunakan spasi
            toks = line.split()
            
            # Jika jumlah token kurang memadai, coba split dengan koma
            if len(toks) < 5:
                toks = line.split(',')
            
            if len(toks) < 5:
                raise ValueError(f"{path}: baris {i} tidak memiliki token minimal (ditemukan {len(toks)} token)")
            
            try:
                cls = int(toks[0])
            except Exception as e:
                raise ValueError(f"{path}: baris {i} error parsing class_id: {e}")
            
            try:
                # Ambil 4 nilai setelah class_id untuk koordinat bounding box
                coords = list(map(float, toks[1:5]))
            except Exception as e:
                raise ValueError(f"{path}: baris {i} error parsing bounding box: {e}")
            
            # Jika ada sisa token, anggap sebagai data segmentasi (polygon)
            seg = None
            if len(toks) > 5:
                try:
                    seg = list(map(float, toks[5:]))
                except Exception as e:
                    raise ValueError(f"{path}: baris {i} error parsing segmen polygon: {e}")
            
            # Validasi nilai koordinat
            if any(np.isnan(coords)) or any(np.isinf(coords)):
                raise ValueError(f"{path}: baris {i} terdapat nilai NaN/Inf pada bounding box")
            if any(c < 0 or c > 1 for c in coords):
                raise ValueError(f"{path}: baris {i} nilai bounding box tidak berada dalam rentang [0,1]")
            
            # Jika ada segmen, validasi jumlah token dan nilai-nilainya
            if seg is not None:
                if len(seg) % 2 != 0 or len(seg) < 6:
                    raise ValueError(f"{path}: baris {i} segmen polygon tidak valid (token tidak genap atau kurang dari 3 titik)")
                if any(np.isnan(seg)) or any(np.isinf(seg)):
                    raise ValueError(f"{path}: baris {i} terdapat nilai NaN/Inf pada segmen")
                if any(s < 0 or s > 1 for s in seg):
                    raise ValueError(f"{path}: baris {i} nilai segmen tidak berada dalam rentang [0,1]")
            
            objs.append({'class': cls, 'bbox': coords, 'seg': seg})
    return objs

def validate_split(img_dir, lbl_dir, stats):
    """
    Melakukan validasi pada satu split (train/val) dengan memeriksa:
      - Keterpaduan file gambar dan label
      - File label kosong
      - Kualitas gambar (bisa dibaca, mendapatkan dimensi)
      - Duplikat gambar (berdasarkan hash MD5)
      - Validitas anotasi: bounding box dalam batas gambar dan validitas polygon
    """
    imgs = sorted(glob.glob(os.path.join(img_dir, '*')))
    lbls = sorted(glob.glob(os.path.join(lbl_dir, '*.txt')))

    img_map = {os.path.splitext(os.path.basename(p))[0]: p for p in imgs}
    lbl_map = {os.path.splitext(os.path.basename(p))[0]: p for p in lbls}

    # Cek ketidakcocokan antara gambar dan label
    missing_lbl = set(img_map) - set(lbl_map)
    missing_img = set(lbl_map) - set(img_map)
    stats['missing_label'] += len(missing_lbl)
    stats['missing_image'] += len(missing_img)
    if missing_lbl:
        print(f"  >> {len(missing_lbl)} gambar tanpa label contoh: {list(missing_lbl)[:5]}")
    if missing_img:
        print(f"  >> {len(missing_img)} label tanpa gambar contoh: {list(missing_img)[:5]}")

    # Cek label kosong
    for k in set(lbl_map) & set(img_map):
        p = lbl_map[k]
        if os.path.getsize(p) == 0:
            stats['empty_label'] += 1

    # Proses membaca gambar dan memeriksa anotasi
    for name, p in tqdm(img_map.items(), desc=f"  Reading images in {os.path.basename(img_dir)}"):
        try:
            img = cv2.imread(p)
            if img is None:
                stats['bad_image'] += 1
                continue
            h, w = img.shape[:2]
            stats['img_sizes'].add((w, h))
            stats['hashes'][md5(p)].append(p)
            if name in lbl_map:
                objs = parse_label(lbl_map[name])
                for obj in objs:
                    # Konversi bounding box dari nilai normalisasi ke pixel
                    cx, cy, bw, bh = obj['bbox']
                    x1 = (cx - bw/2) * w
                    y1 = (cy - bh/2) * h
                    x2 = (cx + bw/2) * w
                    y2 = (cy + bh/2) * h
                    if not (0 <= x1 < w and 0 <= x2 <= w and 0 <= y1 < h and 0 <= y2 <= h):
                        stats['bbox_oob'] += 1
                    # Jika ada data segmentasi, periksa polygon
                    if obj['seg']:
                        pts = np.array(obj['seg']).reshape(-1, 2)
                        pts_px = np.column_stack([pts[:,0] * w, pts[:,1] * h])
                        poly = Polygon(pts_px)
                        if not poly.is_valid:
                            stats['poly_invalid'] += 1
                        if poly.area == 0:
                            stats['poly_zero_area'] += 1
                stats['total_labels'] += len(objs)
                stats['classes'].update([o['class'] for o in objs])
        except Exception as e:
            print(f"  Error loading {p}: {e}")
    return

def report(stats):
    """
    Mencetak laporan ringkas hasil validasi dataset.
    """
    print("\n=== SUMMARY ===")
    print(f"Total gambar bermasalah (tidak bisa dibuka): {stats['bad_image']}")
    print(f"Gambar tanpa label: {stats['missing_label']}")
    print(f"Label tanpa gambar: {stats['missing_image']}")
    print(f"Label kosong: {stats['empty_label']}")
    print(f"Total anotasi: {stats['total_labels']}")
    print(f"Jumlah bbox out-of-bounds: {stats['bbox_oob']}")
    print(f"Jumlah polygon invalid (self-intersect/invalid): {stats['poly_invalid']}")
    print(f"Jumlah polygon area=0: {stats['poly_zero_area']}")
    print(f"Kelas unik ({len(stats['classes'])}): {sorted(stats['classes'])}")
    print("Distribusi kelas:")
    for cls, cnt in stats['classes'].most_common():
        print(f"  - Class {cls}: {cnt}")
    print("Ukuran gambar unik (w,h):", stats['img_sizes'])
    dups = {h: ps for h, ps in stats['hashes'].items() if len(ps) > 1}
    print(f"Jumlah grup duplikat gambar: {len(dups)}")
    if dups:
        for h, ps in list(dups.items())[:3]:
            print(f"  >> hash {h}: {ps[:3]} ...")
    if stats['classes']:
        mx = max(stats['classes'])
        missing_idx = set(range(mx + 1)) - set(stats['classes'].keys())
        print("Indeks kelas terlewat:", sorted(missing_idx) if missing_idx else "Tidak ada")

# Inisialisasi statistik
stats = {
    'bad_image': 0,
    'missing_label': 0,
    'missing_image': 0,
    'empty_label': 0,
    'total_labels': 0,
    'bbox_oob': 0,
    'poly_invalid': 0,
    'poly_zero_area': 0,
    'classes': Counter(),
    'img_sizes': set(),
    'hashes': defaultdict(list),
}

# Validasi dataset untuk setiap split (misal: train dan val)
for split in ['train', 'val']:
    print(f"\n-- Validating split: {split} --")
    img_dir = os.path.join(root_dir, 'images', split)
    lbl_dir = os.path.join(root_dir, 'labels', split)
    validate_split(img_dir, lbl_dir, stats)

report(stats)


-- Validating split: train --


  Reading images in train:  25%|██▌       | 3299/13120 [00:38<01:34, 103.39it/s]

  Error loading C:/AI/sampah_sungai/Dataset_final/Dataset_yolo_Segmentation_v3\images\train\CNG_af7a4image_story_jpg.rf.1824f319c13253bb12062289cacb001e.jpg: C:/AI/sampah_sungai/Dataset_final/Dataset_yolo_Segmentation_v3\labels\train\CNG_af7a4image_story_jpg.rf.1824f319c13253bb12062289cacb001e.txt: baris 4 segmen polygon tidak valid (token tidak genap atau kurang dari 3 titik)
  Error loading C:/AI/sampah_sungai/Dataset_final/Dataset_yolo_Segmentation_v3\images\train\DNAS5W2IMJGB7OMYYFQ4XZQ7GA_jpg.rf.c7255f6327442adcbf3c40c98f5d7a76.jpg: C:/AI/sampah_sungai/Dataset_final/Dataset_yolo_Segmentation_v3\labels\train\DNAS5W2IMJGB7OMYYFQ4XZQ7GA_jpg.rf.c7255f6327442adcbf3c40c98f5d7a76.txt: baris 2 segmen polygon tidak valid (token tidak genap atau kurang dari 3 titik)


  Reading images in train:  65%|██████▌   | 8536/13120 [01:41<00:54, 84.04it/s] 


KeyboardInterrupt: 

In [5]:
import os
import cv2
import numpy as np
import shutil
from tqdm import tqdm
import random

# ✅ Path utama
base_dir = r"C:\AI\sampah_sungai\Dataset_workshop\bengkel\sampah"
label_dir = r"C:\AI\sampah_sungai\Dataset_workshop\bengkel\Cek"
save_dir = r"C:\AI\sampah_sungai\Dataset_workshop\bengkel\Cek"

image_dirs = [
    os.path.join(base_dir, 'images', 'val'),
    os.path.join(base_dir, 'images', 'train')
]
label_dirs = [
    os.path.join(label_dir, 'labels', 'val'),
    os.path.join(label_dir, 'labels', 'train')
]

# ✅ Output dirs
box_out_dir = os.path.join(save_dir, 'box')
mask_out_dir = os.path.join(save_dir, 'mask')

# Bersihkan & buat folder output
for out_dir in [box_out_dir, mask_out_dir]:
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)

def get_class_color(cls):
    """ Menghasilkan warna acak untuk setiap kelas """
    random.seed(cls)  # Seed warna agar konsisten untuk setiap kelas
    return tuple(random.randint(0, 255) for _ in range(3))

# ✅ Mapping label kelas
class_names = {
    0: "sampah",
    1: "orang"
}

def draw_annotations(img, annotations, img_w, img_h):
    boxes = []
    masks = []

    for ann in annotations:
        parts = ann.strip().split()
        if len(parts) < 2:
            continue

        cls = int(parts[0])
        nums = list(map(float, parts[1:]))

        if len(nums) == 4:
            # ➤ Bounding box
            x, y, w, h = nums
            x1 = int((x - w / 2) * img_w)
            y1 = int((y - h / 2) * img_h)
            x2 = int((x + w / 2) * img_w)
            y2 = int((y + h / 2) * img_h)
            boxes.append((cls, x1, y1, x2, y2))
        elif len(nums) >= 6 and len(nums) % 2 == 0:
            # ➤ Polygon
            pts = []
            for i in range(0, len(nums), 2):
                px = int(nums[i] * img_w)
                py = int(nums[i+1] * img_h)
                pts.append((px, py))
            masks.append((cls, pts))

    return boxes, masks

# Kumpulkan semua pasangan label-gambar
label_image_pairs = []
for img_dir, label_dir in zip(image_dirs, label_dirs):
    for label_file in os.listdir(label_dir):
        if not label_file.endswith(".txt"):
            continue

        label_path = os.path.join(label_dir, label_file)

        # Cari gambar yang sesuai (jpg atau png)
        image_file = label_file.replace(".txt", ".jpg")
        image_path = os.path.join(img_dir, image_file)
        if not os.path.exists(image_path):
            image_file = label_file.replace(".txt", ".png")
            image_path = os.path.join(img_dir, image_file)
            if not os.path.exists(image_path):
                continue

        label_image_pairs.append((label_path, image_path, label_file))

# Proses dengan tqdm
count_box_img = 0
count_mask_img = 0

print("🔍 Memproses file anotasi dan gambar...\n")

for label_path, image_path, label_file in tqdm(label_image_pairs, desc="Processing", unit="file"):
    with open(label_path, 'r') as f:
        lines = f.readlines()

    img = cv2.imread(image_path)
    if img is None:
        continue

    img_h, img_w = img.shape[:2]
    boxes, masks = draw_annotations(img, lines, img_w, img_h)

    out_img = img.copy()
    has_box = False
    has_mask = False

    if boxes:
        has_box = True
        for cls, x1, y1, x2, y2 in boxes:
            color = get_class_color(cls)  # Mendapatkan warna berdasarkan kelas
            label_text = class_names.get(cls, str(cls))
            cv2.rectangle(out_img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(out_img, label_text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    if masks:
        has_mask = True
        for cls, pts in masks:
            color = get_class_color(cls)  # Mendapatkan warna berdasarkan kelas
            label_text = class_names.get(cls, str(cls))
            cv2.polylines(out_img, [np.array(pts, dtype=np.int32)], isClosed=True, color=color, thickness=2)
            if pts:
                cv2.putText(out_img, label_text, pts[0], cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    # Simpan hasil visualisasi
    if has_box:
        save_path = os.path.join(box_out_dir, label_file.replace(".txt", ".jpg"))
        cv2.imwrite(save_path, out_img)
        count_box_img += 1

    if has_mask:
        save_path = os.path.join(mask_out_dir, label_file.replace(".txt", ".jpg"))
        cv2.imwrite(save_path, out_img)
        count_mask_img += 1

# Output akhir
print("\n" + "="*60)
print("✅ HASIL CEK ANOTASI (VISUAL GRAFIS YANG DISIMPAN)")
print("="*60)
print(f"🖼️  Total gambar dengan bounding box : {count_box_img}")
print(f"🖌️  Total gambar dengan segmentasi   : {count_mask_img}")

if count_box_img > count_mask_img:
    print("\n📦 Dominan: Gambar Box")
elif count_mask_img > count_box_img:
    print("\n🖍️ Dominan: Gambar Mask")
else:
    print("\n⚖️ Jumlah gambar box dan mask sama")

🔍 Memproses file anotasi dan gambar...



Processing: 100%|██████████| 19976/19976 [04:51<00:00, 68.44file/s] 


✅ HASIL CEK ANOTASI (VISUAL GRAFIS YANG DISIMPAN)
🖼️  Total gambar dengan bounding box : 0
🖌️  Total gambar dengan segmentasi   : 18914

🖍️ Dominan: Gambar Mask


In [1]:
import os
from PIL import Image
from tqdm import tqdm

# ===============================
# Konfigurasi Path
# ===============================
base_dir        = r"C:\AI\sampah_sungai\Dataset_workshop\bengkel\sampah"
save_dir        = r"C:\AI\sampah_sungai\Dataset_workshop\bengkel\Cek"

splits          = ['train', 'val']
label_dirs      = [os.path.join(base_dir, 'labels',  s) for s in splits]
# Pastikan folder images sesuai struktur
image_dirs      = [os.path.join(base_dir, 'images',  s) for s in splits]
save_label_dirs = [os.path.join(save_dir,  'labels', s) for s in splits]

# Buat folder target jika belum ada
for path in save_label_dirs:
    os.makedirs(path, exist_ok=True)

# ===============================
# Fungsi Konversi Segmentasi ke Box
# ===============================
def segment_to_box(coords, img_w, img_h):
    points = []
    for i in range(0, len(coords), 2):
        try:
            x = float(coords[i]) * img_w
            y = float(coords[i+1]) * img_h
            points.append((x, y))
        except (ValueError, IndexError):
            continue
    # Jika titik kurang dari 2, asumsi kotak penuh
    if len(points) < 2:
        return 0.5, 0.5, 1.0, 1.0
    xs, ys = zip(*points)
    x_min = max(0, min(xs)); x_max = min(img_w, max(xs))
    y_min = max(0, min(ys)); y_max = min(img_h, max(ys))
    cx = (x_min + x_max) / 2.0 / img_w
    cy = (y_min + y_max) / 2.0 / img_h
    w  = (x_max - x_min) / img_w
    h  = (y_max - y_min) / img_h
    return cx, cy, w, h

# ===============================
# Proses Utama Konversi
# ===============================
total_files  = {s: 0 for s in splits}
total_lines  = {s: 0 for s in splits}

for split, lbl_dir, img_dir, out_dir in zip(splits, label_dirs, image_dirs, save_label_dirs):
    files = [f for f in os.listdir(lbl_dir) if f.endswith('.txt')]
    for fn in tqdm(files, desc=f"Proses {split}", ncols=80):
        label_path = os.path.join(lbl_dir, fn)
        img_base   = fn.replace('.txt', '')
        # Cari gambar .jpg, .png, .jpeg
        img_path = None
        for ext in ('.jpg', '.png', '.jpeg'):
            candidate = os.path.join(img_dir, img_base + ext)
            if os.path.exists(candidate):
                img_path = candidate
                break
        if not img_path:
            print(f"[WARN] Gambar tidak ditemukan untuk {fn}")
            continue

        try:
            with Image.open(img_path) as img:
                w, h = img.size
        except Exception as e:
            print(f"[ERROR] Buka gambar gagal ({fn}): {e}")
            continue

        new_lines = []
        for line in open(label_path, 'r').read().splitlines():
            total_lines[split] += 1
            parts = line.strip().split()
            if len(parts) < 5:
                # Data kurang → kotak penuh
                cls_id, cx, cy, bw, bh = (parts[0] if parts else '0'), 0.5, 0.5, 1.0, 1.0
            else:
                cls_id, coords = parts[0], parts[1:]
                if len(coords) > 4:
                    cx, cy, bw, bh = segment_to_box(coords, w, h)
                else:
                    try:
                        cx, cy, bw, bh = map(float, coords)
                    except ValueError:
                        # parsing gagal → kotak penuh
                        cx, cy, bw, bh = 0.5, 0.5, 1.0, 1.0
            # Clamp nilai ke [0,1]
            cx, cy = min(max(cx,0),1), min(max(cy,0),1)
            bw, bh = min(max(bw,0),1), min(max(bh,0),1)
            new_lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        # Simpan hasil konversi
        os.makedirs(out_dir, exist_ok=True)
        with open(os.path.join(out_dir, fn), 'w') as f_out:
            f_out.write("\n".join(new_lines))
        total_files[split] += 1

# ===============================
# Ringkasan Konversi
# ===============================
print("\n=== Ringkasan Konversi ===")
for s in splits:
    print(f"{s.capitalize():5s}: {total_files[s]} file, {total_lines[s]} baris diubah ke bbox")

# ===============================
# Validasi Akhir Format Label Kotak (YOLO)
# ===============================
print("\n=== Validasi Akhir ===")
for split, out_dir in zip(splits, save_label_dirs):
    masih_segment = []
    files = [f for f in os.listdir(out_dir) if f.endswith('.txt')]
    for fn in tqdm(files, desc=f"Validasi {split}", ncols=80):
        for line in open(os.path.join(out_dir, fn), 'r').read().splitlines():
            if len(line.strip().split()) > 5:
                masih_segment.append(fn)
                break
    if masih_segment:
        print(f"[WARNING] Masih ada file segmentasi di {out_dir}:")
        for f in set(masih_segment):
            print(f"  - {f}")
    else:
        print(f"[OK] Semua file di {out_dir} sudah dalam format kotak.")

Proses val: 100%|███████████████████████████| 945/945 [00:00<00:00, 1806.15it/s]



=== Ringkasan Konversi ===
Train: 10241 file, 26587 baris diubah ke bbox
Val  : 945 file, 2539 baris diubah ke bbox

=== Validasi Akhir ===


Validasi train: 100%|████████████████████| 10241/10241 [01:15<00:00, 135.35it/s]


[OK] Semua file di C:\AI\sampah_sungai\Dataset_workshop\bengkel\Cek\labels\train sudah dalam format kotak.


Validasi val: 100%|██████████████████████████| 945/945 [00:06<00:00, 142.29it/s]

[OK] Semua file di C:\AI\sampah_sungai\Dataset_workshop\bengkel\Cek\labels\val sudah dalam format kotak.


In [4]:
import os
from PIL import Image
from tqdm import tqdm

# ===============================
# Konfigurasi Path
# ===============================
base_dir = r"C:\AI\sampah_sungai\Dataset_workshop\bengkel\sampah"
save_dir = r"C:\AI\sampah_sungai\Dataset_workshop\bengkel\Cek"

image_dirs = [
    os.path.join(base_dir, 'images', 'train'),
    os.path.join(base_dir, 'images', 'val')
]
label_dirs = [
    os.path.join(base_dir, 'labels', 'train'),
    os.path.join(base_dir, 'labels', 'val')
]
save_label_dirs = [
    os.path.join(save_dir, 'labels', 'train'),
    os.path.join(save_dir, 'labels', 'val')
]

for path in save_label_dirs:
    os.makedirs(path, exist_ok=True)

# ===============================
# Fungsi Konversi Box ke Segmen (4 titik)
# ===============================
def box_to_segment(xc, yc, w, h, img_w, img_h, close_polygon=True):
    box_w, box_h = w * img_w, h * img_h
    cx, cy = xc * img_w, yc * img_h

    points = [
        (cx - box_w / 2, cy - box_h / 2),  # kiri atas
        (cx + box_w / 2, cy - box_h / 2),  # kanan atas
        (cx + box_w / 2, cy + box_h / 2),  # kanan bawaha
        (cx - box_w / 2, cy + box_h / 2),  # kiri bawah
    ]

    if close_polygon:
        points.append(points[0])  # Tambahkan titik awal di akhir (opsional)

    # Normalisasi dan clamp ke [0, 1]
    normalized = []
    for x, y in points:
        x = max(0.0, min(1.0, x / img_w))
        y = max(0.0, min(1.0, y / img_h))
        normalized.extend([x, y])

    return normalized

# ===============================
# Proses Konversi
# ===============================
total_converted = [0, 0]
error_files = []

for i in range(2):  # 0 = train, 1 = val
    label_dir = label_dirs[i]
    image_dir = image_dirs[i]
    save_label_dir = save_label_dirs[i]

    label_files = [f for f in os.listdir(label_dir) if f.endswith('.txt')]
    for filename in tqdm(label_files, desc=f"Processing {'train' if i == 0 else 'val'}", ncols=80):

        label_path = os.path.join(label_dir, filename)
        image_path_jpg = os.path.join(image_dir, filename.replace('.txt', '.jpg'))
        image_path_png = os.path.join(image_dir, filename.replace('.txt', '.png'))

        image_path = image_path_jpg if os.path.exists(image_path_jpg) else image_path_png
        if not os.path.exists(image_path):
            print(f"[WARNING] Gambar tidak ditemukan untuk {filename}")
            continue

        try:
            with Image.open(image_path) as img:
                img_w, img_h = img.size
        except Exception as e:
            print(f"[ERROR] Gagal membuka gambar {filename}: {e}")
            continue

        new_lines = []
        try:
            with open(label_path, 'r') as f:
                lines = f.read().splitlines()
        except Exception as e:
            print(f"[ERROR] Gagal membaca label {filename}: {e}")
            continue

        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try:
                cls = int(parts[0])
                if len(parts) == 5:
                    _, xc, yc, w, h = map(float, parts)
                    seg = box_to_segment(xc, yc, w, h, img_w, img_h)

                    # Cek apakah ada NaN/inf
                    if any([not (0 <= p <= 1) or str(p) == 'nan' or str(p) == 'inf' for p in seg]):
                        raise ValueError("Terdeteksi nilai tidak valid dalam segmentasi.")

                    seg_line = f"{cls} " + " ".join(f"{p:.6f}" for p in seg)
                    new_lines.append(seg_line)
                else:
                    # Sudah segmentasi
                    new_lines.append(line.strip())
            except Exception as e:
                print(f"[ERROR] Kesalahan saat parsing baris di {filename}: {e}")
                error_files.append(filename)
                break

        save_path = os.path.join(save_label_dir, filename)
        with open(save_path, 'w') as f:
            f.write("\n".join(new_lines))

        total_converted[i] += 1

# ===============================
# Ringkasan Proses
# ===============================
print(f"\n[INFO] {total_converted[0]} file (train) berhasil diproses.")
print(f"[INFO] {total_converted[1]} file (val) berhasil diproses.")
if error_files:
    print(f"[WARNING] {len(error_files)} file bermasalah dan dilewati:")
    for f in error_files:
        print(f"  - {f}")

# ===============================
# Validasi Global (Di Akhir)
# ===============================
print("\n=== Validasi Format Label Segmentasi ===")
for i in range(2):
    save_label_dir = save_label_dirs[i]
    still_box_files = []

    label_files = [f for f in os.listdir(save_label_dir) if f.endswith('.txt')]
    for filename in tqdm(label_files, desc=f"Validasi {'train' if i == 0 else 'val'}", ncols=80):
        file_path = os.path.join(save_label_dir, filename)
        with open(file_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    still_box_files.append(filename)
                    break

    if still_box_files:
        print(f"[WARNING] Masih ada file dengan format box di {save_label_dir}:")
        for f in still_box_files:
            print(f"  - {f}")
    else:
        print(f"[OK] Semua file di {save_label_dir} sudah dalam format segmentasi.")

Processing val: 100%|███████████████████████| 3293/3293 [00:48<00:00, 68.40it/s]



[INFO] 16683 file (train) berhasil diproses.
[INFO] 3293 file (val) berhasil diproses.

=== Validasi Format Label Segmentasi ===


Validasi train: 100%|████████████████████| 16683/16683 [01:47<00:00, 155.88it/s]


[OK] Semua file di C:\AI\sampah_sungai\Dataset_workshop\bengkel\Cek\labels\train sudah dalam format segmentasi.


Validasi val: 100%|████████████████████████| 3293/3293 [00:21<00:00, 151.35it/s]

[OK] Semua file di C:\AI\sampah_sungai\Dataset_workshop\bengkel\Cek\labels\val sudah dalam format segmentasi.


In [8]:
import os
import yaml
from collections import defaultdict

# Path utama dataset
base_path = r"C:\AI\sampah_sungai\Dataset_workshop\sampah\deteksi sampah final.v5-export-dataset.yolov11"
yaml_path = os.path.join(base_path, "data.yaml")

# Load nama kelas dari data.yaml
with open(yaml_path, 'r') as f:
    data_yaml = yaml.safe_load(f)

class_names = data_yaml.get('names', [])

# Subfolder dataset
subfolders = ["train", "valid", "test"]
label_counts = defaultdict(int)

# Baca semua file label
for subfolder in subfolders:
    label_dir = os.path.join(base_path, subfolder, "labels")
    
    if not os.path.exists(label_dir):
        print(f"Folder tidak ditemukan: {label_dir}")
        continue

    for file_name in os.listdir(label_dir):
        if file_name.endswith(".txt"):
            file_path = os.path.join(label_dir, file_name)
            with open(file_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        class_id = int(parts[0])
                        label_counts[class_id] += 1

# Tampilkan hasil
print("📦 Statistik Kelas dalam Dataset:")
for class_id in sorted(label_counts.keys()):
    class_name = class_names[class_id] if class_id < len(class_names) else f"ID_{class_id}"
    print(f"- {class_name} (ID {class_id}): {label_counts[class_id]} kemunculan")

Folder tidak ditemukan: C:\AI\sampah_sungai\Dataset_workshop\bengkel\sampah(1)\train\labels
Folder tidak ditemukan: C:\AI\sampah_sungai\Dataset_workshop\bengkel\sampah(1)\valid\labels
Folder tidak ditemukan: C:\AI\sampah_sungai\Dataset_workshop\bengkel\sampah(1)\test\labels
📦 Statistik Kelas dalam Dataset:


In [5]:
import os

# Direktori utama dataset
base_dir = r"C:\AI\sampah_sungai\Dataset_workshop\sampah\deteksi sampah final.v5-export-dataset.yolov11"

# Subdirektori yang berisi label
subfolders = ['train', 'test', 'valid']

# Hitung file yang dimodifikasi
modified_files = 0
deleted_labels = 0

for folder in subfolders:
    label_dir = os.path.join(base_dir, folder, 'labels')
    for label_file in os.listdir(label_dir):
        if label_file.endswith('.txt'):
            file_path = os.path.join(label_dir, label_file)
            with open(file_path, 'r') as f:
                lines = f.readlines()

            # Filter semua baris yang bukan id 0 (person)
            new_lines = [line for line in lines if not line.startswith('0 ')]

            # Jika ada perubahan, tulis ulang file
            if len(new_lines) != len(lines):
                with open(file_path, 'w') as f:
                    f.writelines(new_lines)
                modified_files += 1
                deleted_labels += len(lines) - len(new_lines)

print(f"Selesai. {modified_files} file dimodifikasi.")
print(f"Total {deleted_labels} label 'person' (ID 0) dihapus.")

Selesai. 0 file dimodifikasi.
Total 0 label 'person' (ID 0) dihapus.


In [1]:
import os
from tqdm import tqdm

# Daftar folder label
paths = [
    r"C:\AI\sampah_sungai\Dataset_workshop\bengkel\sampah\labels\train",
    r"C:\AI\sampah_sungai\Dataset_workshop\bengkel\sampah\labels\val"
]

def ubah_label_ke_nol(file_path):
    with open(file_path, "r") as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) >= 5:
            parts[0] = "0"  # Ubah class_id jadi 0
            new_line = " ".join(parts)
            new_lines.append(new_line)

    with open(file_path, "w") as f:
        f.write("\n".join(new_lines) + "\n")

# Proses semua file dengan progress bar
for folder in paths:
    print(f"🔄 Memproses folder: {folder}")
    label_files = [f for f in os.listdir(folder) if f.endswith(".txt")]
    for filename in tqdm(label_files, desc=f"Memproses {os.path.basename(folder)}"):
        full_path = os.path.join(folder, filename)
        ubah_label_ke_nol(full_path)

print("✅ Semua label berhasil diubah menjadi 0.")

🔄 Memproses folder: C:\AI\sampah_sungai\Dataset_workshop\bengkel\sampah\labels\train


Memproses train: 100%|██████████| 16683/16683 [01:52<00:00, 147.65it/s]


🔄 Memproses folder: C:\AI\sampah_sungai\Dataset_workshop\bengkel\sampah\labels\val


Memproses val: 100%|██████████| 3293/3293 [00:27<00:00, 121.95it/s]

✅ Semua label berhasil diubah menjadi 0.


In [1]:
import os
from pathlib import Path
from tqdm import tqdm

# Folder gambar dan label
image_dirs = [
    r"C:\AI\sampah_sungai\Dataset_final\Dataset_yolo_Segmentation_v2\images\train",
    r"C:\AI\sampah_sungai\Dataset_final\Dataset_yolo_Segmentation_v2\images\val"
]

label_dirs = [
    r"C:\AI\sampah_sungai\Dataset_final\Dataset_yolo_Segmentation_v2\labels\train",
    r"C:\AI\sampah_sungai\Dataset_final\Dataset_yolo_Segmentation_v2\labels\val"
]

image_extensions = [".jpg", ".jpeg", ".png"]

def hapus_data_tidak_valid(image_dir, label_dir):
    print(f"\n📂 Memeriksa folder: {image_dir}")
    total_dihapus = 0

    # Hapus gambar jika label tidak ada atau kosong
    for image_file in tqdm(os.listdir(image_dir), desc="🔍 Mengecek gambar"):
        image_path = os.path.join(image_dir, image_file)
        image_name = Path(image_file).stem
        label_path = os.path.join(label_dir, image_name + ".txt")

        if Path(image_path).suffix.lower() in image_extensions:
            if not os.path.isfile(label_path) or is_label_kosong(label_path):
                try:
                    os.remove(image_path)
                    if os.path.exists(label_path):
                        os.remove(label_path)
                    total_dihapus += 1
                except Exception as e:
                    print(f"Gagal menghapus {image_file}: {e}")

    # Hapus label jika gambar tidak ada
    for label_file in tqdm(os.listdir(label_dir), desc="🔍 Mengecek label"):
        label_path = os.path.join(label_dir, label_file)
        label_name = Path(label_file).stem

        # Cari file gambar yang cocok
        ada_gambar = any(os.path.isfile(os.path.join(image_dir, label_name + ext)) for ext in image_extensions)

        if not ada_gambar:
            try:
                os.remove(label_path)
                total_dihapus += 1
            except Exception as e:
                print(f"Gagal menghapus label {label_file}: {e}")

    print(f"🗑️ Total gambar/label dihapus dari {os.path.basename(image_dir)}: {total_dihapus}")

def is_label_kosong(label_path):
    """ Memeriksa apakah file label kosong atau tidak memiliki anotasi yang valid """
    with open(label_path, 'r') as file:
        lines = file.readlines()
        lines = [line.strip() for line in lines if line.strip()]
        return len(lines) == 0

# Proses semua folder gambar dan label
for img_dir, lbl_dir in zip(image_dirs, label_dirs):
    hapus_data_tidak_valid(img_dir, lbl_dir)

print("\n✅ Selesai! Semua gambar & label yang tidak valid telah dihapus.")


📂 Memeriksa folder: C:\AI\sampah_sungai\Dataset_final\Dataset_yolo_Segmentation_v2\images\train


🔍 Mengecek label: 100%|██████████| 15848/15848 [00:00<00:00, 56679.98it/s]


🗑️ Total gambar/label dihapus dari train: 835

📂 Memeriksa folder: C:\AI\sampah_sungai\Dataset_final\Dataset_yolo_Segmentation_v2\images\val


🔍 Mengecek label: 100%|██████████| 3066/3066 [00:00<00:00, 57169.88it/s]

🗑️ Total gambar/label dihapus dari val: 227

✅ Selesai! Semua gambar & label yang tidak valid telah dihapus.


In [1]:
import os

# Path folder
labels_path = r"C:\AI\sampah_sungai\Dataset_workshop\bengkel\sampah(1)\labels\train"
images_path = r"C:\AI\sampah_sungai\Dataset_workshop\bengkel\sampah(1)\images\train"

# Dapatkan nama file tanpa ekstensi
label_files = {os.path.splitext(f)[0] for f in os.listdir(labels_path) if f.endswith(".txt")}
image_files = {os.path.splitext(f)[0] for f in os.listdir(images_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))}

# Hapus gambar yang tidak punya label
for img_file in image_files:
    if img_file not in label_files:
        img_full_path = os.path.join(images_path, img_file + ".jpg")  # Ganti dengan ekstensi lain jika perlu
        if not os.path.exists(img_full_path):
            img_full_path = os.path.join(images_path, img_file + ".png")  # fallback
        if os.path.exists(img_full_path):
            os.remove(img_full_path)
            print(f"Hapus gambar tanpa label: {img_full_path}")

# Hapus label yang tidak punya gambar
for lbl_file in label_files:
    if lbl_file not in image_files:
        lbl_full_path = os.path.join(labels_path, lbl_file + ".txt")
        os.remove(lbl_full_path)
        print(f"Hapus label tanpa gambar: {lbl_full_path}")

In [1]:
from ultralytics import YOLO

# Load a model
model = YOLO(r"C:\AI\sampah_sungai\Hasil\yolo11m-seg(v1.2)\weights\best.pt")  # load a custom trained model

# Export the model
model.export(format="onnx")

Ultralytics 8.3.109  Python-3.12.7 torch-2.6.0+cu124 CPU (11th Gen Intel Core(TM) i9-11950H 2.60GHz)
YOLO11m-seg summary (fused): 138 layers, 22,336,854 parameters, 0 gradients, 123.0 GFLOPs

PyTorch: starting from 'C:\AI\sampah_sungai\Hasil\yolo11m-seg(v1.2)\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 38, 8400), (1, 32, 160, 160)) (43.0 MB)

ONNX: starting export with onnx 1.17.0 opset 19...
ONNX: slimming with onnxslim 0.1.50...
ONNX: export success  4.2s, saved as 'C:\AI\sampah_sungai\Hasil\yolo11m-seg(v1.2)\weights\best.onnx' (85.5 MB)

Export complete (5.7s)
Results saved to C:\AI\sampah_sungai\Hasil\yolo11m-seg(v1.2)\weights
Predict:         yolo predict task=segment model=C:\AI\sampah_sungai\Hasil\yolo11m-seg(v1.2)\weights\best.onnx imgsz=640  
Validate:        yolo val task=segment model=C:\AI\sampah_sungai\Hasil\yolo11m-seg(v1.2)\weights\best.onnx imgsz=640 data=C:\AI\sampah_sungai\Dataset_final\Dataset_yolo_Segmentation_v3\dataset.yaml  
V

'C:\\AI\\sampah_sungai\\Hasil\\yolo11m-seg(v1.2)\\weights\\best.onnx'

In [ ]:
import os
from pathlib import Path
import cv2
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm
import warnings
import logging
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

# ---- CONFIGURATION ----
model_path = Path(r"C:\AI\sampah_sungai\Hasil\yolo11x-cls(v3)\weights\best.pt")
dataset_root = Path(r"C:\AI\sampah_sungai\Dataset_workshop\bengkel\sampah(1)")
output_root = Path(r"C:\AI\sampah_sungai\Dataset_workshop\bengkel\Cek\label_predict")
splits = ['train', 'val']
class_names = [
    'Glass', 'Metal', 'Plastic', 'Plastic_Buoy', 'Styrofoam', 'twig', 'wood'
]

# Silence ultralytics logging to keep output clean
logging.getLogger('ultralytics').setLevel(logging.ERROR)

# Load YOLO model (classification only)
model = YOLO(str(model_path))

# Ensure output directories exist
for split in splits:
    (output_root / split).mkdir(parents=True, exist_ok=True)

# Build a mapping from image stem to full path for display
img_map = {}
for split in splits:
    for img_path in (dataset_root / 'images' / split).glob('*.*'):
        img_map[img_path.stem] = img_path

# Process each split with tqdm progress bars
for split in splits:
    images_dir = dataset_root / 'images' / split
    labels_dir = dataset_root / 'labels' / split
    out_dir = output_root / split

    img_paths = list(images_dir.glob('*.*'))
    for img_path in tqdm(img_paths, desc=f'Processing {split}', unit='img'):
        stem = img_path.stem
        label_file = labels_dir / f"{stem}.txt"
        if not label_file.exists():
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            continue

        new_lines = []
        with open(label_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 3:
                    continue
                orig_cls = parts[0]
                coords = list(map(float, parts[1:]))
                pts = np.array(coords, dtype=np.float32).reshape(-1, 2)

                # Compute bounding box for the polygon and crop
                x, y, w, h = cv2.boundingRect(pts.astype(np.int32))
                crop = img[y:y+h, x:x+w]
                if crop.size == 0:
                    continue

                # Predict class on the crop
                results = model(crop)
                if results and results[0].boxes and len(results[0].boxes.cls) > 0:
                    pred_cls = int(results[0].boxes.cls[0])
                else:
                    # fallback to original if no detection
                    pred_cls = int(orig_cls)

                # Append new annotation (classification only)
                new_lines.append(f"{pred_cls} {' '.join(parts[1:])}")

        # Save predictions
        out_file = out_dir / f"{stem}.txt"
        with open(out_file, 'w') as fw:
            fw.write("\n".join(new_lines))

# Single warning upon completion
display_completed = warnings.warn("Selesai proses")

# Display sample annotated images
for split in splits:
    txt_files = list((output_root / split).glob('*.txt'))[:3]
    for txt_file in txt_files:
        stem = txt_file.stem
        img_path = img_map.get(stem)
        if img_path is None:
            continue
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(6, 6))
        plt.imshow(img_rgb)
        ax = plt.gca()

        with open(txt_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                cls = int(parts[0])
                coords = np.array(list(map(float, parts[1:])), dtype=np.float32).reshape(-1, 2)
                poly = Polygon(coords, closed=True, fill=False, linewidth=2, edgecolor='r')
                ax.add_patch(poly)
                ax.text(coords[0, 0], coords[0, 1], class_names[cls], fontsize=10,
                        color='yellow', backgroundcolor='black')

        plt.title(f"{split} - {stem}")
        plt.axis('off')
        plt.show()

In [ ]:
#-----------------------------------------------------------#
############# STEP 02 : Cek hasil Augmentasi ##############
#-----------------------------------------------------------#

import cv2              # Untuk manipulasi gambar
import numpy as np      # Untuk operasi array dan numerik
import random           # Untuk pengacakan (misal: memilih gambar acak, warna)
import os               # Untuk operasi file dan folder
import glob             # Untuk pencarian file dengan pola tertentu
import logging
from tqdm import tqdm   # Untuk progress bar di terminal
import shutil

# ============================================
# Konfigurasi Logging
# ============================================
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.handlers.clear()  # Hapus handler default

# Handler untuk menyimpan log ke file (INFO ke atas)
file_handler = logging.FileHandler("proses.log")
file_handler.setLevel(logging.INFO)
file_formatter = logging.Formatter('%(asctime)s %(levelname)s: %(message)s')
file_handler.setFormatter(file_formatter)
logger.addHandler(file_handler)

# Handler untuk menampilkan log ke terminal (WARNING ke atas)
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.WARNING)
console_formatter = logging.Formatter('%(levelname)s: %(message)s')
console_handler.setFormatter(console_formatter)
logger.addHandler(console_handler)

# ============================================
# Definisi Folder Sumber dan Tujuan
# ============================================
aug_img_dir = r"C:\AI\sampah_sungai\Dataset_workshop\bengkel\sampah(1)\images\train"
aug_label_dir = r"C:\AI\sampah_sungai\Dataset_workshop\bengkel\label_predict\train"
save_dir = r"C:\AI\sampah_sungai\Dataset_workshop\bengkel\Cek"

# ============================================
# Pastikan Folder Tujuan Ada dan Kosong
# ============================================
if not os.path.exists(save_dir):
    os.makedirs(save_dir, exist_ok=True)
    logger.info("Folder '%s' dibuat.", save_dir)
else:
    # Kosongkan folder cek jika sudah ada
    for filename in os.listdir(save_dir):
        file_path = os.path.join(save_dir, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            logger.warning("Gagal menghapus %s. Error: %s", file_path, e)
    logger.info("Folder '%s' dikosongkan.", save_dir)

# ============================================
# Pengambilan File Gambar
# ============================================
img_files = glob.glob(os.path.join(aug_img_dir, "*.jpg"))

if len(img_files) < 1000000:
    logger.warning("Gambar kurang dari 100! Menampilkan semua yang ada.")
    random_imgs = img_files
else:
    random_imgs = random.sample(img_files, 100)

logger.info("Menampilkan %d gambar untuk pengecekan.", len(random_imgs))

# ============================================
# Fungsi Bantuan
# ============================================

def compute_polygon_area(pts):
    """Menghitung luas poligon menggunakan metode contourArea OpenCV."""
    return cv2.contourArea(pts)

def is_polygon_within_bounds(pts, width, height):
    """Memeriksa apakah semua titik poligon berada dalam batas gambar."""
    for pt in pts.reshape(-1, 2):
        x, y = pt
        if x < 0 or x > width or y < 0 or y > height:
            return False
    return True

def round_polygon(pts, precision=4):
    """Mengembalikan tuple dari koordinat poligon yang sudah dibulatkan."""
    return tuple(np.round(pts.flatten(), precision))

# ============================================
# Proses Pengolahan dan Pengecekan Tiap Gambar
# ============================================
for img_file in tqdm(random_imgs, desc="Memproses gambar"):
    base_name = os.path.splitext(os.path.basename(img_file))[0]
    label_file = os.path.join(aug_label_dir, base_name + ".txt")
    
    if not os.path.exists(label_file):
        logger.warning("Label untuk %s tidak ditemukan, lewati.", base_name)
        continue
    
    image = cv2.imread(img_file)
    if image is None:
        logger.warning("Gagal membaca %s, lewati.", img_file)
        continue
    
    h, w, _ = image.shape
    original_image = image.copy()  # Untuk menggambar anotasi
    
    with open(label_file, "r") as f:
        lines = f.readlines()
    
    # Dictionary untuk mendeteksi duplikasi: key = (class_id, rounded koordinat)
    seen_polygons = {}
    duplicate_found = False
    
    for idx, line in enumerate(lines):
        parts = line.strip().split()
        if len(parts) < 3:
            logger.warning("Format label salah pada %s baris %d.", base_name, idx+1)
            continue

        # Parsing label dan koordinat
        class_id = parts[0]
        try:
            coords = np.array(parts[1:], dtype=np.float32).reshape(-1, 2)
        except Exception as e:
            logger.warning("Gagal parsing koordinat pada %s baris %d. Error: %s", base_name, idx+1, e)
            continue
        
        # Konversi koordinat relatif ke piksel
        coords[:, 0] *= w
        coords[:, 1] *= h
        coords = coords.astype(np.int32)
        pts = coords.reshape((-1, 1, 2))
        
        # Periksa apakah poligon berada dalam batas gambar
        if not is_polygon_within_bounds(pts, w, h):
            logger.warning("Poligon pada %s baris %d berada di luar batas gambar.", base_name, idx+1)
            # Tandai dengan warna oranye
            color = (0, 165, 255)
            cv2.polylines(image, [pts], isClosed=True, color=color, thickness=3)
        else:
            # Tandai dengan warna hijau jika valid
            color = (0, 255, 0)
            cv2.polylines(image, [pts], isClosed=True, color=color, thickness=2)
        
        # Periksa area poligon
        area = compute_polygon_area(pts)
        if area < 10:
            logger.warning("Area poligon terlalu kecil (%.2f) pada %s baris %d.", area, base_name, idx+1)
            # Tandai dengan warna biru
            cv2.polylines(image, [pts], isClosed=True, color=(255, 0, 0), thickness=3)
        
        # Cek duplikasi label
        key = (class_id, round_polygon(pts, precision=2))
        if key in seen_polygons:
            logger.warning("Duplikasi label ditemukan pada %s baris %d. Duplikat dengan baris %d.", 
                           base_name, idx+1, seen_polygons[key])
            duplicate_found = True
            # Tandai duplikasi dengan warna merah
            cv2.polylines(image, [pts], isClosed=True, color=(0, 0, 255), thickness=3)
        else:
            seen_polygons[key] = idx+1  # Simpan nomor baris label
        
    # Simpan gambar hasil pengecekan
    save_path = os.path.join(save_dir, base_name + "_checked.jpg")
    cv2.imwrite(save_path, image)
    
    logger.info("Gambar %s telah dicek dan disimpan di %s", base_name + "_checked.jpg", save_dir)
    
    # Jika ditemukan duplikasi, juga simpan gambar asli untuk referensi
    if duplicate_found:
        dup_save_path = os.path.join(save_dir, base_name + "_duplicate.jpg")
        cv2.imwrite(dup_save_path, original_image)
        logger.info("Gambar asli %s juga disimpan sebagai referensi duplikasi.", base_name)

logger.warning("Proses pengecekan selesai!")

In [1]:
import os
import shutil
import random
from PIL import Image
from tqdm import tqdm

names = ['Glass', 'Metal', 'Plastic', 'Plastic_Buoy', 'Styrofoam', 'twig', 'wood']
nc = len(names)

source_base = r"C:\AI\sampah_sungai\Dataset_workshop\bengkel\sampah"
dest_base = r"C:\AI\sampah_sungai\Dataset_final\Dataset_yolo_Clasification_v1"

MIN_SIZE = 20  # ukuran crop minimum (pixel)

# Step 1: konversi dari deteksi ke klasifikasi
for split in ['train', 'val']:
    images_dir = os.path.join(source_base, "images", split)
    labels_dir = os.path.join(source_base, "labels", split)

    image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    for filename in tqdm(image_files, desc=f"Processing {split}"):
        img_path = os.path.join(images_dir, filename)
        label_path = os.path.join(labels_dir, os.path.splitext(filename)[0] + '.txt')

        if not os.path.exists(label_path):
            continue

        try:
            image = Image.open(img_path).convert("RGB")
        except:
            continue

        w, h = image.size

        with open(label_path, 'r') as f:
            lines = [line.strip() for line in f if line.strip()]

        if len(lines) == 1:
            class_id = int(lines[0].split()[0])
            class_name = names[class_id]

            out_dir = os.path.join(dest_base, split, class_name)
            os.makedirs(out_dir, exist_ok=True)
            shutil.copy(img_path, os.path.join(out_dir, filename))
        else:
            for i, line in enumerate(lines):
                parts = line.split()
                try:
                    class_id = int(parts[0])
                    x_center, y_center, bw, bh = map(float, parts[1:5])
                except ValueError:
                    continue

                class_name = names[class_id]
                x_center *= w
                y_center *= h
                bw *= w
                bh *= h
                x1 = int(x_center - bw / 2)
                y1 = int(y_center - bh / 2)
                x2 = int(x_center + bw / 2)
                y2 = int(y_center + bh / 2)

                # Perbesar jika terlalu kecil
                if (x2 - x1) < MIN_SIZE:
                    pad = (MIN_SIZE - (x2 - x1)) // 2
                    x1 -= pad
                    x2 += pad
                if (y2 - y1) < MIN_SIZE:
                    pad = (MIN_SIZE - (y2 - y1)) // 2
                    y1 -= pad
                    y2 += pad

                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(w, x2)
                y2 = min(h, y2)

                if x2 <= x1 or y2 <= y1:
                    continue  # tetap abaikan jika tidak valid

                cropped = image.crop((x1, y1, x2, y2))

                out_dir = os.path.join(dest_base, split, class_name)
                os.makedirs(out_dir, exist_ok=True)
                new_name = f"{os.path.splitext(filename)[0]}_{i}.jpg"
                try:
                    cropped.save(os.path.join(out_dir, new_name))
                except:
                    print(f"⚠️  Gagal simpan crop: {filename}_{i}")

print("\n🟢 Konversi selesai. Sekarang menyeimbangkan dataset val...")

# Step 2: Seimbangkan val agar semua class ada dan jumlahnya merata
val_dir = os.path.join(dest_base, "val")
train_dir = os.path.join(dest_base, "train")

# Hitung jumlah gambar per kelas di val
val_counts = {}
for class_name in names:
    class_path = os.path.join(val_dir, class_name)
    if os.path.exists(class_path):
        val_counts[class_name] = len(os.listdir(class_path))
    else:
        val_counts[class_name] = 0
        os.makedirs(class_path)

max_val = max([c for c in val_counts.values() if c > 0])  # target agar semua class seimbang

# Tambahkan gambar dari train ke val jika class val-nya kosong atau kurang
for class_name in names:
    need = max_val - val_counts[class_name]
    if need > 0:
        train_path = os.path.join(train_dir, class_name)
        val_path = os.path.join(val_dir, class_name)

        available = os.listdir(train_path)
        random.shuffle(available)
        moved = 0

        for f in available:
            src = os.path.join(train_path, f)
            dst = os.path.join(val_path, f)
            if not os.path.exists(dst):
                shutil.copy(src, dst)
                moved += 1
                if moved >= need:
                    break

        print(f"✅ Menambahkan {moved} gambar ke val/{class_name}")

print("\n✅ Dataset val sekarang seimbang di 7 kelas.")

Processing val: 100%|██████████| 945/945 [00:06<00:00, 150.59it/s]



🟢 Konversi selesai. Sekarang menyeimbangkan dataset val...
✅ Menambahkan 692 gambar ke val/Glass
✅ Menambahkan 1055 gambar ke val/Metal
✅ Menambahkan 136 gambar ke val/Plastic_Buoy
✅ Menambahkan 994 gambar ke val/Styrofoam
✅ Menambahkan 905 gambar ke val/twig
✅ Menambahkan 825 gambar ke val/wood

✅ Dataset val sekarang seimbang di 7 kelas.
